In [ ]:
import pygame as pg
from pygame import *
import sys
import random

In [ ]:
white = (255, 255, 255)
red = (255, 0, 0)
green = (0, 255, 0)
blue = (0, 0, 255)
black = (0, 0, 0)

In [ ]:
# Setup
pg.init()

# Other variables
WIDTH = 400
HEIGHT = 450
ACC = 0.5
FRIC = -0.12
FPS = 60

vec = pg.math.Vector2

# Setting up fonts
font = pg.font.SysFont("Verdana", 60)
font_small = pg.font.SysFont("Verdana", 20)
game_over = font.render("Game Over", True, white)
pg.display.set_caption('Game')

# Classes
class Player(pg.sprite.Sprite):
    def __init__(self, game):
        super().__init__()
        self.game = game

        self.surf = pg.Surface((30, 30))
        self.surf.fill((128, 255, 40))
        
        self.rect = self.surf.get_rect(center = (10, 420))

        self.pos = vec((10, 360))
        self.vel = vec(0,0)
        self.acc = vec(0,0)

        self.jumping = False
    
    def move(self):
        self.acc = vec(0, 0.5)
 
        pressed_keys = pg.key.get_pressed()
            
        if pressed_keys[K_LEFT] or pressed_keys[K_a]:
            self.acc.x = -ACC
        if pressed_keys[K_RIGHT] or pressed_keys[K_d]:
            self.acc.x = ACC

        self.acc.x += self.vel.x * FRIC
        self.vel += self.acc
        self.pos += self.vel + 0.5 * self.acc

        if self.pos.x > WIDTH:
            self.pos.x = 0
        if self.pos.x < 0:
            self.pos.x = WIDTH
     
        self.rect.midbottom = self.pos

    def jump(self):
        hits = pg.sprite.spritecollide(
            self,
            self.game.platforms,
            False
        )
        print(hits)
        if hits and not self.jumping:
            self.jumping = True

            self.vel.y = -15
    
    def cancel_jump(self):
        if self.jumping:
            if self.vel.y < -3:
                self.vel.y = -3

    # Função de limitação por colisão
    def update(self):
        hits = pg.sprite.spritecollide(
            self,
            self.game.platforms,
            False
        )
        
        if self.vel.y > 0 and hits:
            platform = hits[0]

            if self.pos.y < platform.rect.bottom:
                self.vel.y = 0
                self.pos.y = platform.rect.top + 1
                self.jumping = False
                self.pos.x += platform.vel.x
                # print("RESET")
    def draw(self, screen):
        screen.blit(self.surf, self.rect)

class Platform(pg.sprite.Sprite):
    def __init__(self, width_max=100, moving=False, speed=0):
        super().__init__()

        self.width_max = width_max
        self.moving = moving
        self.speed = speed
        self.direction = random.randint(-1, 1)
        
        self.surf = pg.Surface(
            (
                random.randint(
                    int(width_max/2),
                    int(width_max)
                ),
                15
            )
        )

        self.surf.fill(green)

        coordenadas = (
            random.randint(0, WIDTH-10),
            random.randint(0, HEIGHT-20)
        )
        self.rect = self.surf.get_rect(
            center = (coordenadas)
        )

        self.vel = vec(speed, 0)
        self.pos = vec(coordenadas)

    def move(self):
        if self.moving:
            self.pos.x += self.vel.x*self.direction

            if self.pos.x <= 0:
                self.direction = 1
            elif self.pos.x >= WIDTH:
                self.direction = -1
            
            self.rect.x = self.pos.x
                
    def draw(self, screen):
        screen.blit(self.surf, self.rect)

class MinorPlatform(Platform):
    def __init__(self):
        super().__init__(
            width_max=50,
            moving=True,
            speed=3
        )

class MajorPlatform(Platform):
    def __init__(self):
        super().__init__(
            width_max=80,
            moving=True,
            speed=6
        )

# Game loop
class Game:
    def __init__(self):
        self.running = True

        self.screen = pg.display.set_mode((WIDTH, HEIGHT))
        self.clock = pg.time.Clock()
        self.camera_offset = 0
        self.world_height = 0
        self.camera_run = False

        self.all_sprites = pg.sprite.Group()
        self.platforms = pg.sprite.Group()

        self.setup()

    def setup(self):
        self.player = Player(self)
        base_platform = Platform()

        base_platform.surf = pg.Surface((WIDTH, 20))
        base_platform.surf.fill(red)

        base_platform.rect = base_platform.surf.get_rect(
            center=(WIDTH/2, WIDTH-10)
        )
        
        self.platforms.add(base_platform)
        
        self.all_sprites.add(base_platform)
        self.all_sprites.add(self.player)

        for _ in range(5):
            while True:
                pl = Platform()
                # Se a plataforma for correta...
                if not self.check_colision(pl):
                    break

            self.platforms.add(pl)
            self.all_sprites.add(pl)

    def check_colision(self, platform):
        if pg.sprite.spritecollideany(platform, self.platforms):
            return True
        else:
            for entity in self.platforms:
                if entity == platform:
                    continue
                if (
                    abs(platform.rect.top - entity.rect.bottom) < 50
                    and
                    (abs(platform.rect.bottom - entity.rect.top) < 50)
                ):
                    return True
    
        return False

    def plat_gen(self):
        while len(self.platforms) < 6:
            while True:

                if self.world_height <= 1000:
                    PlatformClass = Platform
                elif self.world_height > 1000 and self.world_height <= 3000:
                    PlatformClass = random.choice(
                        [Platform, MinorPlatform]
                    )
                elif self.world_height > 3000 and self.world_height <= 9000:
                    PlatformClass = random.choice(
                        [Platform, MinorPlatform, MajorPlatform]
                    )
                else:
                    chances = [1, 3, 6]
                    PlatformClass = random.choices(
                        [Platform, MinorPlatform, MajorPlatform],
                        weights=chances
                    )[0]

                p = PlatformClass()
                p.rect.center = (
                    random.randrange(0, WIDTH),
                    -(random.randrange(0, 50))
                )

                p.pos = vec(p.rect.center)

                if (
                    not self.check_colision(p)
                    or
                    p.rect.top - self.player.rect.bottom > 80
                ):
                    break

            self.platforms.add(p)
            self.all_sprites.add(p)

    def input(self):
        for event in pg.event.get():
            if event.type == QUIT:
                self.running = False
            if event.type == pg.KEYDOWN:
                if event.key == pg.K_SPACE:
                    self.player.jump()
            if event.type == pg.KEYUP:
                if event.key == pg.K_SPACE:
                    self.player.cancel_jump()

    def update(self):
        self.player.move()
        self.player.update()

        for plat in self.platforms:
            plat.move()

        # Camera
            # Se o player ultrapassar 2/3 da tela
        if self.player.rect.top <= HEIGHT/3:
            self.camera_run = True
            self.camera_offset = abs(self.player.vel.y)
            self.world_height += self.camera_offset
            self.player.pos.y += self.camera_offset

            for plat in self.platforms:
                plat.move()
                plat.rect.y += self.camera_offset

                # Remove plataformas fora da tela
                if plat.rect.top >= HEIGHT:
                    plat.kill()
        
        elif self.camera_run:
            self.camera_offset = 1
            self.world_height += self.camera_offset
            self.player.pos.y += self.camera_offset

            for plat in self.platforms:
                plat.move()
                plat.rect.y += self.camera_offset

                # Remove plataformas fora da tela
                if plat.rect.top >= HEIGHT:
                    plat.kill()
        
        self.plat_gen()

        # Game Over
        if self.player.rect.top >= HEIGHT:
            self.running = False

    def draw(self):
        self.screen.fill(black)
        text_height = font_small.render(
            f"Height: {int(self.world_height)}",
            True,
            white
        )

        self.screen.blit(text_height, (10, 10))

        for entity in self.all_sprites:
            entity.draw(self.screen)
        
        pg.display.update()

    def run(self):
        while self.running:
            self.clock.tick(FPS)
            self.input()
            self.update()
            self.draw()
        
        pg.quit()
        sys.exit()

game = Game()
game.run()